# Task 7: Low-Rank Adaptation (LoRA) Matrix Projection Fine-Tuning of a 3B Model

## Formula
$$W^\prime = W_0 + \frac{\alpha}{r} (B \cdot A)$$


In [ ]:
import torch
import torch.nn as nn

# LoRA wrapper layer inserting low-rank matrices A and B parallel to frozen linear weights
class LoRALinear(nn.Module):
    def __init__(self, in_features, out_features, r=4, lora_alpha=8):
        super().__init__()
        self.linear = nn.Linear(in_features, out_features)
        self.linear.weight.requires_grad = False # Freeze base parameters
        
        self.scaling = lora_alpha / r
        self.lora_A = nn.Parameter(torch.randn(r, in_features) * 0.01)
        self.lora_B = nn.Parameter(torch.zeros(out_features, r))

    def forward(self, x):
        return self.linear(x) + (x @ self.lora_A.T @ self.lora_B.T) * self.scaling


In [ ]:
# Instantiate LoRA layer and verify parameter reduction ratio
layer = LoRALinear(in_features=128, out_features=128, r=4)
trainable = sum(p.numel() for p in layer.parameters() if p.requires_grad)
frozen = sum(p.numel() for p in layer.parameters() if not p.requires_grad)

print(f"Trainable Parameters: {trainable} | Frozen Parameters: {frozen}")
print(f"Trainable Ratio: {100 * trainable / (trainable + frozen):.2f}%")
print("[SUCCESS] LoRA adapter layer verified.")
